In [1]:
import torch
import torch.nn.functional as F

import pandas as pd

import preprocess
import update_model
import validate

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

DEVICE = get_device()
print('Using device:', str(DEVICE).upper(), "\n")

MODEL_DIR="./model"
DATA_PATH = "war_and_peace.txt"

TRAIN_DATA_BUCKET = "cbow-training-data-1f656"
MODEL_DATA_BUCKET = "cbow-model-data-1f656"

In [ ]:
# # === Run this code for the first model initialization ===

# # Download training data
# preprocess.download_data_from_gcs(
#     TRAIN_DATA_BUCKET,
#     DATA_PATH,
# )

# model, cfg = update_model.init_first_model(data_path=DATA_PATH)

In [ ]:
# === Run this code to initialize pretrained model ===

# Download training data
preprocess.download_data_from_gcs(
    TRAIN_DATA_BUCKET,
    DATA_PATH
)

# Download pretrained model and its configuration
preprocess.download_model_from_gcs(
    bucket_name=MODEL_DATA_BUCKET,
    download_dir=MODEL_DIR,
)

# Initialize pretrained model
model, cfg = update_model.init_model(
    new_data_path=f"data/{DATA_PATH}"
)

In [ ]:
# === Train model ===
model = update_model.train(
    model=model,
    cfg=cfg,
    device=DEVICE,
    epochs=50,
)

# === Save model configuration locally ===
update_model.save(model, cfg)

In [26]:
# # === Save model configuration in GCS and create new version ===
# preprocess.save_model_to_gcs(bucket_name=MODEL_DATA_BUCKET)

In [20]:
model = model.to(DEVICE)

word_embeddings = model.get_input_embeddings() # Shape: (vocab_size, embedding_dim)
word_embeddings = F.normalize(word_embeddings, p=2, dim=1)

window_size = cfg['window_size']
word2idx    = cfg["word2idx"]
idx2word    = {i: w for w, i in word2idx.items()}

In [22]:
print("\n" + "="*40)

test_word = "green"
top_k = 5
print(f"Top {top_k} similar words to '{test_word}':")
res = validate.find_similar_words(
    test_word, 
    word_embeddings, 
    word2idx, 
    idx2word,
    k=top_k
)

for w, score in res:
    print(f"  {w:15} {score:.4f}")
print("="*40)


Top 5 similar words to 'green':
  blue            0.3795
  light           0.3424
  unix            0.3397
  samovar         0.3253
  oil             0.3223


In [23]:
word_1 = "he"
word_2 = "they"

v1 = word_embeddings[word2idx[word_1]]
v2 = word_embeddings[word2idx[word_2]]
print(f"Original: {F.cosine_similarity(v1, v2, dim=0).item()}")

Original: 0.40151017904281616


In [13]:
words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

In [24]:
word_embeddings = model.get_input_embeddings()

embedding_size = cfg['embedding_shape'][1]
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

# thresholds = [0.1,0.2,0.3,0.4]
thresholds = [i for i in range(1, 11)]

for t in thresholds:
    
    # filtering
    B = (embeddings >= t).int()
    # C = torch.cov(B.T)
    C = B @ B.T

    print(f" -- threshold = {t}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_1-cov_matrix_{t}.csv")

 -- threshold = 1, nonzoer(B) = 451
 -- threshold = 2, nonzoer(B) = 56
 -- threshold = 3, nonzoer(B) = 2
 -- threshold = 4, nonzoer(B) = 0
 -- threshold = 5, nonzoer(B) = 0
 -- threshold = 6, nonzoer(B) = 0
 -- threshold = 7, nonzoer(B) = 0
 -- threshold = 8, nonzoer(B) = 0
 -- threshold = 9, nonzoer(B) = 0
 -- threshold = 10, nonzoer(B) = 0


In [25]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

deltas = [0.01, 0.02, 0.03] # [0.02,0.05]

for d in deltas:
    # filtering
    B = (torch.abs(embeddings) <= d).int()
    C = B @ B.T

    print(f"-- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_2-cov_matrix_{d}.csv")

-- delta = 0.01, nonzoer(B) = 38
-- delta = 0.02, nonzoer(B) = 68
-- delta = 0.03, nonzoer(B) = 105
